# Test if pesticide usage is a useful predictor

Compare baseline model to models that have the baseline features and:

1. feature engineered 2 (respiratory) and 6 (chemical class aggregate) pesticide features
2. 30+ features that survive Lasso regression
3. principle components that explain 90% of the variance
4. all 400+ individual pesticide compounds

In [1]:
import os
import numpy as np
import pandas as pd

import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt

In [2]:
DATA_DIR = "../data"

TRAIN_CASTHMA_PATH = os.path.join(DATA_DIR, "train_CASTHMA.csv")

TRAIN_COPD_PATH = os.path.join(DATA_DIR, "train_COPD.csv")

train_asthma = pd.read_csv(TRAIN_CASTHMA_PATH)
train_copd = pd.read_csv(TRAIN_COPD_PATH)

In [3]:
DEMO_COLS = [
    'population','median_age','median_income',
    'pct_white','pct_black','pct_asian','pct_hispanic',
    'rural_binary']
HEALTH_CONFOUNDER_COLS = ['CSMOKING','OBESITY','DIABETES']
CROPLAND_COLS = ['cropland_diversity','county_crop_concentration','pct_cropland']
OTHER_COLS = ['YEAR']

BASE_COLS = DEMO_COLS + HEALTH_CONFOUNDER_COLS + CROPLAND_COLS + OTHER_COLS

PEST_AGGREGATE = [
    'log_resp_per_capita',
    'log_resp_per_cropland_acre']

PEST_COMPONENTS = [
    'log_op_per_capita',
    'log_op_per_cropland_acre',
    'log_carbamate_per_capita',
    'log_carbamate_per_cropland_acre',
    'log_pyrethroid_per_capita',
    'log_pyrethroid_per_cropland_acre']

cols = train_asthma.columns[train_asthma.columns.str.startswith("pesticide_")]
X_subset = train_asthma[cols]
X_subset = X_subset.drop(columns=['pesticide_total_kg','pesticide_respiratory_kg',
                                 'pesticide_organochlorine_kg','pesticide_organophosphate_kg',
                                 'pesticide_carbamate_kg','pesticide_pyrethroid_kg',
                                 'pesticide_triazine_kg','pesticide_chlorophenoxy_kg',
                                 'pesticide_anilide_kg'])
PEST_ALL = X_subset.columns

In [4]:
def engineer_features(df):
    df = df.copy()

    df = df.fillna(0)

    df['resp_per_capita'] = df['pesticide_respiratory_kg'] / df['population']
    df['resp_per_cropland_acre'] = df['pesticide_respiratory_kg'] / df['cropland_acres']

    df['op_per_capita'] = df['pesticide_organophosphate_kg'] / df['population']
    df['op_per_cropland_acre'] = df['pesticide_organophosphate_kg'] / df['cropland_acres']

    df['carbamate_per_capita'] = df['pesticide_carbamate_kg'] / df['population']
    df['carbamate_per_cropland_acre'] = df['pesticide_carbamate_kg'] / df['cropland_acres']

    df['pyrethroid_per_capita'] = df['pesticide_pyrethroid_kg'] / df['population']
    df['pyrethroid_per_cropland_acre'] = df['pesticide_pyrethroid_kg'] / df['cropland_acres']

    pest_intensity_cols = [
        'resp_per_capita','resp_per_cropland_acre',
        'op_per_capita', 'op_per_cropland_acre',
        'carbamate_per_capita', 'carbamate_per_cropland_acre',
        'pyrethroid_per_capita', 'pyrethroid_per_cropland_acre']
    for col in pest_intensity_cols:
        df[f'log_{col}'] = np.log1p(df[col])

    df['rural_binary'] = (df['nchs_urban_rural'] >= 5).astype(int)

    # take all pesticide columns excluding the aggregate ones and normalize the raw component kg
    cols = df.columns[df.columns.str.startswith("pesticide_")]
    cols_to_drop = ['pesticide_total_kg','pesticide_respiratory_kg',
                     'pesticide_organochlorine_kg','pesticide_organophosphate_kg',
                     'pesticide_carbamate_kg','pesticide_pyrethroid_kg',
                     'pesticide_triazine_kg','pesticide_chlorophenoxy_kg',
                     'pesticide_anilide_kg']
    cols = [col for col in cols if col not in cols_to_drop] # extracted 400+ individual component list
    for col in cols:
        df[f'log_{col[10:-3]}_per_cap'] = np.log1p(df[col] / df['population'])
        df[f'log_{col[10:-3]}_per_crop'] = np.log1p(df[col] / df['cropland_acres'])
    
    return df
    
train_asthma = engineer_features(train_asthma)
train_copd = engineer_features(train_copd)
    

/tmp/ipykernel_6079/3928423203.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'log_{col[10:-3]}_per_crop'] = np.log1p(df[col] / df['cropland_acres'])
/tmp/ipykernel_6079/3928423203.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'log_{col[10:-3]}_per_cap'] = np.log1p(df[col] / df['population'])
/tmp/ipykernel_6079/3928423203.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at on

In [5]:
def compare_models(df, target, base_predictors, extra_predictors):

    base_formula = target + " ~ " + " + ".join(base_predictors)
    full_formula = target + " ~ " + " + ".join(base_predictors + extra_predictors)

    base_model = smf.ols(base_formula, data=df).fit()
    full_model = smf.ols(full_formula, data=df).fit()

    anova = sm.stats.anova_lm(base_model, full_model)

    # Perform the F-test
    f_test = full_model.compare_f_test(base_model)
    
    # Print the results
    print("F-statistic:", f_test[0])
    print("p-value:", f_test[1])
    if f_test[1] < 0.05:
        print("Full model improves fit")

    return {
        "base_model": base_model,
        "full_model": full_model,
        "anova": anova,
        "F": f_test[0],
        "p": f_test[1],
        "aic": (base_model.aic, full_model.aic),
        "bic": (base_model.bic, full_model.bic)
    }

### 1. Baseline + engineered features for pesticides

In [6]:
base_vs_aggregate_asthma = compare_models(
    train_asthma,
    target="CASTHMA",
    base_predictors=BASE_COLS,
    extra_predictors=PEST_AGGREGATE
)
base_vs_component_asthma = compare_models(
    train_asthma,
    target="CASTHMA",
    base_predictors=BASE_COLS,
    extra_predictors=PEST_AGGREGATE
)

base_vs_aggregate_copd = compare_models(
    train_copd,
    target="COPD",
    base_predictors=BASE_COLS,
    extra_predictors=PEST_COMPONENTS
)
base_vs_component_copd = compare_models(
    train_copd,
    target="COPD",
    base_predictors=BASE_COLS,
    extra_predictors=PEST_COMPONENTS
)

F-statistic: 57.379311209057754
p-value: 2.3472158755019207e-25
Full model improves fit
F-statistic: 57.379311209057754
p-value: 2.3472158755019207e-25
Full model improves fit
F-statistic: 18.669661787908876
p-value: 1.3931534382089713e-21
Full model improves fit
F-statistic: 18.669661787908876
p-value: 1.3931534382089713e-21
Full model improves fit


### 2. Baseline + Lasso-surviving features

In [7]:
# run lasso regression and then select the pesticide-features that survive

### 3. Baseline + PCA on pesticides
Testing only on pesticide per capita for now

In [8]:
cols = train_asthma.columns[train_asthma.columns.str.startswith("pesticide_")]
cols_to_drop = ['pesticide_total_kg','pesticide_respiratory_kg',
                 'pesticide_organochlorine_kg','pesticide_organophosphate_kg',
                 'pesticide_carbamate_kg','pesticide_pyrethroid_kg',
                 'pesticide_triazine_kg','pesticide_chlorophenoxy_kg',
                 'pesticide_anilide_kg','pesticide_other_kg']
PEST_ALL_CAP = [f'log_{col[10:-3]}_per_cap' for col in cols if col not in cols_to_drop] # extracted 400+ individual component list
PEST_ALL_CROP = [f'log_{col[10:-3]}_per_crop' for col in cols if col not in cols_to_drop] # extracted 400+ individual component list

In [9]:
def add_pcs(df):
    df=df.copy()
    X=df[PEST_ALL_CAP]
    
    # Scale
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # PCA
    pca = PCA(n_components=0.9) # cutoff at 90% explained variance
    X_pca = pca.fit_transform(X_scaled)
    X_pca = pd.DataFrame(X_pca)
    X_pca = X_pca.add_prefix('PC_')
    
    df = pd.concat([df,X_pca],axis=1)
    cols = X_pca.columns.tolist()
    
    return df, cols

In [10]:
train_asthma_PCA, PC_COLS = add_pcs(train_asthma)

base_vs_PCs_asthma = compare_models(
    train_asthma_PCA,
    target="CASTHMA",
    base_predictors=BASE_COLS,
    extra_predictors=PC_COLS
)

F-statistic: 8.075620985783496
p-value: 6.044650327508956e-100
Full model improves fit


In [11]:
train_copd_PCA, PC_COLS = add_pcs(train_copd)
base_vs_PCs_copd = compare_models(
    train_copd_PCA,
    target="COPD",
    base_predictors=BASE_COLS,
    extra_predictors=PC_COLS
)

F-statistic: 4.8761828584304325
p-value: 1.5326612101655456e-49
Full model improves fit


### 4. Baseline + 400+ pesticides

In [12]:
base_vs_all_cap_asthma = compare_models(
    train_asthma,
    target="CASTHMA",
    base_predictors=BASE_COLS,
    extra_predictors=PEST_ALL_CAP
)
base_vs_all_crop_asthma = compare_models(
    train_asthma,
    target="CASTHMA",
    base_predictors=BASE_COLS,
    extra_predictors=PEST_ALL_CROP
)

base_vs_all_cap_copd = compare_models(
    train_copd,
    target="COPD",
    base_predictors=BASE_COLS,
    extra_predictors=PEST_ALL_CROP
)
base_vs_all_crop_copd = compare_models(
    train_copd,
    target="COPD",
    base_predictors=BASE_COLS,
    extra_predictors=PEST_ALL_CROP
)

F-statistic: 6.047125743460324
p-value: 2.8271427343810337e-197
Full model improves fit
F-statistic: 6.618636759687439
p-value: 1.2843974868110138e-223
Full model improves fit
F-statistic: 3.2226772561773367
p-value: 1.786934435817008e-73
Full model improves fit
F-statistic: 3.2226772561773367
p-value: 1.786934435817008e-73
Full model improves fit
